# Building and Loading a Software Dependency Graph into Neo4j

We will use the [deps.dev](https://deps.dev/) software dependency dataset, to build a comprehensive graph of software dependencies and load it into a Neo4j database for analysis. We will use the deps.dev public API, and will recursively fetch dependency information for a set of software packages up to a given depth.

In [1]:
# Load .env
from dotenv import load_dotenv
import os

load_dotenv()

NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
SEED_PACKAGES = [
    pkg.strip() for pkg in os.getenv("SEED_PACKAGES", "").split(",") if pkg.strip()
]
SEED_SYSTEM = os.getenv("SEED_SYSTEM", "pypi")
SEED_DEPTH = int(os.getenv("SEED_DEPTH", 2))

In [2]:
import time
import random
import requests
import pandas as pd
import urllib.parse
from tqdm.auto import tqdm


def api_get_with_backoff(
    url: str, max_retries: int = 5, base_delay: float = 5.0
) -> dict:
    """
    Safely handles network calls with exponential backoff.
    Fails fast without retrying on strict client errors like 404.
    """
    delay = base_delay
    for attempt in range(max_retries):
        try:
            response = requests.get(url, timeout=10)

            if response.status_code == 200:
                return response.json()

            if response.status_code in [429, 500, 502, 503, 504]:
                print(
                    f"[Warning] HTTP {response.status_code}. Retrying in {delay:.2f}s... (Attempt {attempt + 1}/{max_retries})"
                )
                time.sleep(delay + random.uniform(0, 0.5))
                delay *= 2
                continue

            response.raise_for_status()

        except requests.exceptions.HTTPError as http_err:
            raise http_err

        except requests.RequestException as net_err:
            print(
                f"[Warning] Connection anomaly ({net_err}). Retrying in {delay:.2f}s... (Attempt {attempt + 1}/{max_retries})"
            )
            time.sleep(delay + random.uniform(0, 0.5))
            delay *= 2

    raise Exception(f"Max retries exceeded for URL: {url}")


def get_latest_package_dependencies_flat(
    system: str, package_name: str, max_depth: int = 3
) -> pd.DataFrame:
    """
    Fetches the pre-resolved dependency tree from deps.dev. If the absolute latest
    version hasn't been fully processed by Google yet, it automatically backtracks
    chronologically to find the most recent version with a populated dependency graph.
    """
    system = system.upper()
    encoded_package = urllib.parse.quote(package_name, safe="")

    # 1. Fetch all package metadata and available versions
    package_url = (
        f"https://api.deps.dev/v3/systems/{system.lower()}/packages/{encoded_package}"
    )
    try:
        package_data = api_get_with_backoff(package_url)
    except Exception as e:
        print(f"[Error] Could not find package metadata for '{package_name}': {e}")
        return pd.DataFrame()

    versions = package_data.get("versions", [])
    if not versions:
        print(f"[Warning] No versions found for {package_name}")
        return pd.DataFrame()

    # 2. Sort versions chronologically (newest first) based on publishedAt timestamp
    versions_sorted = sorted(
        versions, key=lambda x: x.get("publishedAt", ""), reverse=True
    )

    # Build prioritized candidate queue: Default version first, then newest-to-oldest
    candidate_versions = []
    for v in versions:
        if v.get("isDefault"):
            candidate_versions.append(v["versionKey"]["version"])
            break

    for v in versions_sorted:
        v_str = v["versionKey"]["version"]
        if v_str not in candidate_versions:
            candidate_versions.append(v_str)

    # 3. Loop through candidates until Google returns a valid resolved dependency graph
    deps_data = None
    resolved_version = None

    for v_str in candidate_versions:
        deps_url = f"https://api.deps.dev/v3/systems/{system.lower()}/packages/{encoded_package}/versions/{v_str}:dependencies"
        try:
            data = api_get_with_backoff(deps_url)
            # Ensure the graph actually contains resolved nodes and structural edges
            if data.get("nodes") and data.get("edges"):
                deps_data = data
                resolved_version = v_str
                break
        except Exception:
            # If a new version returns 404 or fails to resolve, catch it and try the next version
            continue

    if not deps_data:
        print(
            f"[Warning] deps.dev has not resolved a dependency graph for ANY version of '{package_name}' yet."
        )
        return pd.DataFrame()

    # Log if backtracking occurred
    if resolved_version != candidate_versions[0]:
        print(
            f"--> Note: Absolute latest version ({candidate_versions[0]}) graph is unpopulated in deps.dev. Backtracked to '{resolved_version}'."
        )
    else:
        print(f"--> Using version '{resolved_version}' for '{package_name}'.")

    nodes = deps_data.get("nodes", [])
    edges = deps_data.get("edges", [])

    # 4. Robust Root Node Detection
    # Locate where the requested root package sits inside the response array to ground the BFS
    root_idx = None
    for idx, node in enumerate(nodes):
        node_name = node.get("versionKey", {}).get("name", "")
        if node_name.lower() == package_name.lower():
            root_idx = idx
            break
    if root_idx is None:
        root_idx = 0  # Fallback to index 0 if not explicitly matched

    # 5. Local In-Memory BFS to calculate exact tree depth from the discovered root node
    node_depths = {root_idx: 0}
    queue = [root_idx]

    adj = {}
    for edge in edges:
        f = int(edge.get("fromNode", 0))
        t = int(edge.get("toNode", 0))
        if f not in adj:
            adj[f] = []
        adj[f].append(t)

    head = 0
    while head < len(queue):
        curr = queue[head]
        head += 1
        curr_depth = node_depths[curr]

        if curr_depth >= max_depth:
            continue

        for neighbor in adj.get(curr, []):
            if neighbor not in node_depths:
                node_depths[neighbor] = curr_depth + 1
                queue.append(neighbor)

    # 6. Extract all graph edges matching your depth criteria
    records = []
    for edge in edges:
        f = int(edge.get("fromNode", 0))
        t = int(edge.get("toNode", 0))

        if f in node_depths and node_depths[f] < max_depth:
            source_node = nodes[f].get("versionKey", {})
            target_node = nodes[t].get("versionKey", {})

            records.append(
                {
                    "ecosystem": system,
                    "source_package": source_node.get("name"),
                    "source_version": source_node.get("version"),
                    "target_package": target_node.get("name"),
                    "target_version": target_node.get("version"),
                    "semver_requirement": edge.get("requirement", ""),
                    "graph_depth": node_depths[f] + 1,
                }
            )

    return pd.DataFrame(records)


def crawl_seed_packages_flat(
    system: str, seed_packages: list, max_depth: int = 3
) -> pd.DataFrame:
    """Iterates through seed packages, utilizing single-shot graph extraction."""
    all_dfs = []

    print(f"Starting optimized batch crawl for {len(seed_packages)} packages...\n")
    for package in tqdm(seed_packages, desc="Crawling packages"):
        print(f"--> Fetching full graph for seed: {package}")
        df = get_latest_package_dependencies_flat(system, package, max_depth)

        if not df.empty:
            all_dfs.append(df)
            print(f"    Captured {len(df)} graph edges from the '{package}' tree.")
        else:
            print(f"    Skipping or no edges found for '{package}'.")

    if all_dfs:
        master_df = pd.concat(all_dfs, ignore_index=True)
        master_df.drop_duplicates(
            subset=["ecosystem", "source_package", "source_version", "target_package"],
            inplace=True,
        )
        return master_df
    return pd.DataFrame()


def get_package_version_vulnerabilities(
    system: str, package_name: str, version: str
) -> list:
    """
    Retrieves detailed security advisories (including CVE IDs, titles, and CVSS scores)
    for a specific package version.
    """
    system = system.upper()
    encoded_package = urllib.parse.quote(package_name, safe="")
    encoded_version = urllib.parse.quote(version, safe="")

    version_url = f"https://api.deps.dev/v3/systems/{system.lower()}/packages/{encoded_package}/versions/{encoded_version}"

    try:
        version_data = api_get_with_backoff(version_url)
    except Exception as e:
        # Pass silently or log if a version layout throws errors
        return []

    advisory_keys = version_data.get("advisoryKeys", [])
    vulnerabilities = []

    for adv_key in advisory_keys:
        adv_id = adv_key.get("id")
        if not adv_id:
            continue

        advisory_url = (
            f"https://api.deps.dev/v3/advisories/{urllib.parse.quote(adv_id, safe='')}"
        )
        try:
            adv_data = api_get_with_backoff(advisory_url)

            aliases = adv_data.get("aliases", [])
            title = adv_data.get("title", "")
            url = adv_data.get("url", "")
            cvss_score = adv_data.get("cvss3Score", None)
            cvss_vector = adv_data.get("cvss3Vector", "")

            # Filter or extract formal CVE tracking IDs out of the aliases array
            cve_ids = [alias for alias in aliases if alias.startswith("CVE-")]

            # Fallback: if no CVE alias exists but the advisory ID is a CVE string, adopt it
            if not cve_ids and adv_id.startswith("CVE-"):
                cve_ids = [adv_id]

            # Map out a record for each validated CVE identifier associated with the vulnerability
            for cve in (cve_ids if cve_ids else [adv_id]):
                vulnerabilities.append(
                    {
                        "advisory_id": adv_id,
                        "cve_id": cve,
                        "title": title,
                        "url": url,
                        "cvss3_score": cvss_score,
                        "cvss3_vector": cvss_vector,
                    }
                )

        except Exception as e:
            print(f"[Warning] Failed to resolve advisory details for {adv_id}: {e}")
            # Graceful fallback: append basic identifier metadata if the detail call breaks
            vulnerabilities.append(
                {
                    "advisory_id": adv_id,
                    "cve_id": adv_id,
                    "title": "Advisory details unretrievable",
                    "url": "",
                    "cvss3_score": None,
                    "cvss3_vector": "",
                }
            )

    return vulnerabilities


def extract_vulnerabilities_from_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Inspects a crawled dependency DataFrame, unifies all unique
    source/target package versions, scans them for vulnerabilities,
    and returns a structured Vulnerabilities DataFrame.
    """
    if df.empty:
        return pd.DataFrame()

    # 1. Isolate every unique package version pair found across the graph
    sources = df[["ecosystem", "source_package", "source_version"]].rename(
        columns={"source_package": "package_name", "source_version": "version"}
    )
    targets = df[["ecosystem", "target_package", "target_version"]].rename(
        columns={"target_package": "package_name", "target_version": "version"}
    )

    unique_elements = pd.concat([sources, targets]).drop_duplicates().dropna()
    vuln_records = []

    print(
        f"\nScanning {len(unique_elements)} unique package-versions for security vulnerabilities..."
    )

    for _, row in tqdm(
        unique_elements.iterrows(), total=len(unique_elements), desc="Fetching CVEs"
    ):
        sys = row["ecosystem"]
        pkg = row["package_name"]
        ver = row["version"]

        cves = get_package_version_vulnerabilities(sys, pkg, ver)
        for cve in cves:
            vuln_records.append(
                {
                    "ecosystem": sys,
                    "package_name": pkg,
                    "version": ver,
                    "advisory_id": cve["advisory_id"],
                    "cve_id": cve["cve_id"],
                    "title": cve["title"],
                    "url": cve["url"],
                    "cvss3_score": cve["cvss3_score"],
                    "cvss3_vector": cve["cvss3_vector"],
                }
            )

    return pd.DataFrame(vuln_records)

In [ ]:
from IPython.display import display

# Recursively retrieve dependencies for all packages up to the specified depth
final_graph_df = crawl_seed_packages_flat(
    system=SEED_SYSTEM, seed_packages=SEED_PACKAGES, max_depth=SEED_DEPTH
)

print("=" * 50)
print(f"BATCH CRAWL COMPLETE!")
print(
    f"Total Unique Graph Edges Collected: {len(final_graph_df)}, for depth {SEED_DEPTH}"
)
print("=" * 50)

# Preview the final data structure
display(final_graph_df)

Starting optimized batch crawl for 446 packages...



Crawling packages:   0%|          | 0/446 [00:00<?, ?it/s]

--> Fetching full graph for seed: boto3
--> Note: Absolute latest version (1.43.30) graph is unpopulated in deps.dev. Backtracked to '1.43.27'.
    Captured 8 graph edges from the 'boto3' tree.
--> Fetching full graph for seed: packaging
--> Note: Absolute latest version (26.2.0) graph is unpopulated in deps.dev. Backtracked to '21.3.0'.
    Captured 1 graph edges from the 'packaging' tree.
--> Fetching full graph for seed: urllib3
[Warning] deps.dev has not resolved a dependency graph for ANY version of 'urllib3' yet.
    Skipping or no edges found for 'urllib3'.
--> Fetching full graph for seed: certifi
[Warning] deps.dev has not resolved a dependency graph for ANY version of 'certifi' yet.
    Skipping or no edges found for 'certifi'.
--> Fetching full graph for seed: idna
[Warning] deps.dev has not resolved a dependency graph for ANY version of 'idna' yet.
    Skipping or no edges found for 'idna'.
--> Fetching full graph for seed: requests
--> Using version '2.34.2' for 'requests'

In [ ]:
vulnerabilities_df = extract_vulnerabilities_from_dataframe(final_graph_df)

display(vulnerabilities_df)


Scanning 500 unique package-versions for security vulnerabilities...


Fetching CVEs:   0%|          | 0/500 [00:00<?, ?it/s]

,ecosystem,package_name,version,advisory_id,cve_id,title,url,cvss3_score,cvss3_vector
0,PYPI,torch,2.8.0,GHSA-qfhq-4f3w-5fph,CVE-2025-3001,PyTorch is vulnerable to memory corruption thr...,https://osv.dev/vulnerability/GHSA-qfhq-4f3w-5fph,5.3,CVSS:3.1/AV:L/AC:L/PR:L/UI:N/S:U/C:L/I:L/A:L
1,PYPI,torch,2.8.0,GHSA-rrmf-rvhw-rf47,CVE-2025-3000,PyTorch is vulnerable to memory corruption thr...,https://osv.dev/vulnerability/GHSA-rrmf-rvhw-rf47,5.3,CVSS:3.1/AV:L/AC:L/PR:L/UI:N/S:U/C:L/I:L/A:L
2,PYPI,torch,2.8.0,GHSA-vgrw-7cvw-pwgx,CVE-2025-2999,PyTorch is vulnerable to memory corruption thr...,https://osv.dev/vulnerability/GHSA-vgrw-7cvw-pwgx,5.3,CVSS:3.1/AV:L/AC:L/PR:L/UI:N/S:U/C:L/I:L/A:L
3,PYPI,torch,2.8.0,PYSEC-2025-203,CVE-2025-55551,PYSEC-2025-203,https://osv.dev/vulnerability/PYSEC-2025-203,7.5,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:N/I:N/A:H
4,PYPI,torch,2.8.0,PYSEC-2025-204,CVE-2025-55552,PYSEC-2025-204,https://osv.dev/vulnerability/PYSEC-2025-204,7.5,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:N/I:N/A:H
5,PYPI,torch,2.8.0,PYSEC-2025-206,CVE-2025-55554,PYSEC-2025-206,https://osv.dev/vulnerability/PYSEC-2025-206,5.3,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:N/I:N/A:L
6,PYPI,torch,2.8.0,PYSEC-2026-139,CVE-2026-4538,PYSEC-2026-139,https://osv.dev/vulnerability/PYSEC-2026-139,7.8,CVSS:3.1/AV:L/AC:L/PR:L/UI:N/S:U/C:H/I:H/A:H
7,PYPI,starlette,1.2.1,GHSA-82w8-qh3p-5jfq,CVE-2026-54283,Starlette: request.form() limits silently igno...,https://osv.dev/vulnerability/GHSA-82w8-qh3p-5jfq,7.5,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:N/I:N/A:H
8,PYPI,starlette,1.2.1,GHSA-jp82-jpqv-5vv3,CVE-2026-54282,Starlette: Unvalidated request path concatenat...,https://osv.dev/vulnerability/GHSA-jp82-jpqv-5vv3,3.7,CVSS:3.1/AV:N/AC:H/PR:N/UI:N/S:U/C:N/I:L/A:N
9,PYPI,ray,2.51.2,GHSA-gx77-xgc2-4888,CVE-2025-34351,Ray's New Token Authentication is Disabled By ...,https://osv.dev/vulnerability/GHSA-gx77-xgc2-4888,0.0,


## The graph schema

Our graph schema will be as follows:

`:Package->[:DEPENDS_ON]->:Package`
`:Package->[:HAS_VULNERABILITY]->:Vulnerability`

`:Package` nodes will have the following properties:

- `name`: The name of the package.

While `:Vulnerability` nodes will have the following properties:

- `advisory_id`: The unique identifier for the vulnerability.
- `cve_id`: The Common Vulnerabilities and Exposures (CVE) identifier for the vulnerability.
- `title`: The title or brief description of the vulnerability.
- `url`: The URL with more information about the vulnerability.
- `cvss3_score`: The CVSS v3 score indicating the severity of the vulnerability.
- `cvss3_vector`: The CVSS v3 vector string representing the vulnerability's characteristics.

The `:DEPENDS_ON` relationship will have the following properties:

- `semver_requirement`: The semantic versioning requirement for the dependency.
- `source_version`: The version of the source package.
- `target_version`: The version of the target package.

While the `:HAS_VULNERABILITY` relationship will have the following properties:

- `version`: The version of the package affected by the vulnerability.

In [ ]:
from neo4j import GraphDatabase


def create_constraints(tx):
    """Creates uniqueness constraints for the graph."""
    # Note: 'CREATE CONSTRAINT IF NOT EXISTS' is standard for Neo4j 4.4+
    tx.run(
        "CREATE CONSTRAINT package_name_unique IF NOT EXISTS FOR (p:Package) REQUIRE p.name IS UNIQUE"
    )
    tx.run(
        "CREATE CONSTRAINT vulnerability_id_unique IF NOT EXISTS FOR (v:Vulnerability) REQUIRE v.advisory_id IS UNIQUE"
    )


def load_package_dependencies(tx, records):
    """
    Loads package dependency records into Neo4j safely by isolating
    node merges to distinct sets before drawing relationships.
    """
    # Step 1: Merge all UNIQUE source packages in this batch
    tx.run(
        """
    UNWIND $records AS record
    WITH DISTINCT record.source_package AS pkg_name
    MERGE (:Package {name: pkg_name})
    """,
        records=records,
    )

    # Step 2: Merge all UNIQUE target packages in this batch
    tx.run(
        """
    UNWIND $records AS record
    WITH DISTINCT record.target_package AS pkg_name
    MERGE (:Package {name: pkg_name})
    """,
        records=records,
    )

    # Step 3: Match the now-guaranteed nodes and safely merge the relationship
    query = """
    UNWIND $records AS record
    MATCH (source:Package {name: record.source_package})
    MATCH (target:Package {name: record.target_package})
    MERGE (source)-[r:DEPENDS_ON {
        source_version: record.source_version,
        target_version: record.target_version
    }]->(target)
    SET r.semver_requirement = record.semver_requirement
    """
    tx.run(query, records=records)


def load_vulnerabilities(tx, records):
    """
    Loads vulnerability records safely. Since one CVE can affect multiple
    packages/versions, this prevents duplicate Vulnerability node collisions.
    """
    # Step 1: Ensure all packages in this vulnerability batch exist
    tx.run(
        """
    UNWIND $records AS record
    WITH DISTINCT record.package_name AS pkg_name
    MERGE (:Package {name: pkg_name})
    """,
        records=records,
    )

    # Step 2: Ensure all unique vulnerabilities in this batch are merged cleanly
    tx.run(
        """
    UNWIND $records AS record
    WITH DISTINCT record.advisory_id AS adv_id
    MERGE (:Vulnerability {advisory_id: adv_id})
    """,
        records=records,
    )

    # Step 3: Map properties and draw the edges using safe MATCH lookups
    query = """
    UNWIND $records AS record
    MATCH (source:Package {name: record.package_name})
    MATCH (v:Vulnerability {advisory_id: record.advisory_id})
    MERGE (source)-[r:HAS_VULNERABILITY]->(v)
    SET v.title = record.title,
        v.cve_id = record.cve_id,
        v.url = record.url,
        v.cvss3_score = record.cvss3_score,
        v.cvss3_vector = record.cvss3_vector
    SET r.version = record.version
    """
    tx.run(query, records=records)


# Initialize Driver
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

try:
    # Setup Schema (Constraints)
    with driver.session(database=NEO4J_DATABASE) as session:
        session.execute_write(create_constraints)
        print("[Success] Uniqueness constraints verified/created.")

    # Batch Load Data
    # Convert DataFrame to list of dictionaries for the driver
    package_records = final_graph_df.to_dict("records")

    # Load in batches of 1000 for efficiency
    batch_size = 1000
    for i in range(0, len(package_records), batch_size):
        batch = package_records[i : i + batch_size]
        with driver.session(database=NEO4J_DATABASE) as session:
            session.execute_write(load_package_dependencies, batch)
        print(
            f"[Success] Loaded batch {i // batch_size + 1} ({len(batch)} package records)."
        )

    # Load vulnerabilities
    vulnerability_records = vulnerabilities_df.to_dict("records")
    for i in range(0, len(vulnerability_records), batch_size):
        batch = vulnerability_records[i : i + batch_size]
        with driver.session(database=NEO4J_DATABASE) as session:
            session.execute_write(load_vulnerabilities, batch)
        print(
            f"[Success] Loaded vulnerability batch {i // batch_size + 1} ({len(batch)} vulnerability records)."
        )

    print("\n" + "=" * 50)
    print("GRAPH LOADING COMPLETE!")
    print("=" * 50)

finally:
    driver.close()

[Success] Uniqueness constraints verified/created.
[Success] Loaded batch 1 (1000 package records).
[Success] Loaded batch 2 (345 package records).
[Success] Loaded vulnerability batch 1 (33 vulnerability records).

GRAPH LOADING COMPLETE!
